# 16 - Train Offline DQN with Reward Centering

This notebook follows the same offline workflow as `02_train_offline_dqn.ipynb`, with one extra piece: a **reward-centering constant** learned as a head.

`ConstantHead` is the centering constant. It is a head, and it is a parameter, but `forward` ignores the backbone state and returns `scale * value`. `value` starts at 0, so the output starts at 0. `scale` sets how far one step of `value` moves the output. The DQN objective shapes every reward slot by that constant: a slot whose discount is `γ` subtracts `(1 - γ) c'`, which telescopes to the **same offset `c` on every action value** — the action-value head learns `Q - c` and the policy ordering never changes, episodic or continuing. (Subtracting `c` from every reward directly would penalize longer horizons and reorder policies once episodes terminate.) The loss is `(δ - K c)²` with `δ = G - Q(s, a)` and `K` the target recursion run on reward `1 - γ` and value `0`. Here `γ` is `1.0` in-task and `0` at the task end, so that offset is subtracted exactly once, on the task-final transition. The backup uses the delayed center: one step is `G = r - c' + γ c' + γ V`. The objective returns a dict of losses. `action_value` is the TD loss and trains Q. `reward_center` is `-c · mean(δ)` with `δ` detached, so gradient descent steps `c` by that mean.

The delayed model copies every head. The backup uses the delayed center. The `reward_center` loss trains the online center, and `model_polyak` tracks the delayed copy.

1. Load previously collected `Datastore` streams from the Hub.
2. Build a `DataLoader` that fills one batch (`batch_size=1`) with whole tasks up to `TOKEN_BUDGET` packed tokens.
3. Assemble a `Model` from a pretrained backbone, an action-value head, and a `ConstantHead`.
4. Train by passing `reward_center=out.predictions["reward_center"]` and `delayed_reward_center=delayed_out.predictions["reward_center"]` on the `DqnObjective` call. The delayed model copies every head. Save with `push_model_to_hub`.

This is a short usage example, not a full experiment. Evaluate a saved checkpoint in `15_inference.ipynb`.


In [ ]:
import torch

from mouse_core import AdamW
from mouse_core.data import (
    to_device,
    DataLoader,
    Augmenter,
    Tokenizer,
    compose,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective, boundary_discount
from mouse_core.models import Model, model_polyak, push_model_to_hub
from mouse_core.models.backbone import TransformerBackbone
from mouse_core.models.heads import ConstantHead, RegressionHead

DATASET_ID = "mouse-example-dataset"                               # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-model-reward-centering-offline"          # Hugging Face model repo for push_model_to_hub
TOKENIZER_ID = "mouse-example-tokenizer-reward-centering-offline"  # Hugging Face tokenizer repo (separate from MODEL_ID)
PRETRAINED = "Qwen/Qwen3-0.6B"                                     # HF checkpoint for Tokenizer and backbone
MAX_ACTIONS = 4                                                    # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                                              # vocabulary size for discrete observations
MAX_STEPS_PER_EPISODE = 30                                         # max steps per episode (matches collect)
MAX_TASK_EPISODES = 20                                             # max episodes per task (matches collect)
TOKEN_BUDGET = 16384                                               # packed tokens; whole tasks are added until the next one would not fit
NUM_CYCLES = 2                                                     # outer train cycles (print cadence)
TRAIN_STEPS = 50                                                   # optimizer updates per cycle (passed to run_train)
POLYAK_TAU_HEADS = 0.0001                                          # delayed Q-head interpolation (0 = frozen, 1 = copy of the online heads)
POLYAK_TAU_BACKBONE = 0.01                                         # delayed backbone interpolation

LR_BACKBONE = 1e-5                                                 # AdamW lr for the backbone
LR_HEADS = 1e-5                                                    # AdamW lr for the heads

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(repo_id=DATASET_ID, split="train", force_download=True)


## Data Pipeline

`DataLoader` samples contiguous runs of one `sample_field` from one or more datastores. These examples pass `sample_field="task_index"`, so **each packed group is one task** (every contiguous row that shares a `task_index`). Pass exactly one of `token_budget` or `samples_budget`, plus `batch_size`. `batch_size` is how many times that budget is filled, and `1` is a single fill. Each fill adds whole tasks until the next task would pass the budget. `token_budget` counts packed tokens (step tokens plus each task's group-start tokens). `samples_budget` counts steps in the task. A task that does not fit is left out, and each kept task gets a unique group id. The first task of a fill is left out too when it alone is over the budget, and that raises. These examples pass `token_budget=TOKEN_BUDGET`.

Pipeline order: `augmenter → tokenizer → pack → backbone.embed`.

| Stage | Role |
| --- | --- |
| **Augmenter** | `dict → dict` (`fields=` value transforms; `seed_field=` shares draws within one sampled sequence). Each window gets its own `reseed` generation, so the same index on two rollouts draws two seeds. Action permute sets `input_vector_field` / `output_vector_field` on `info_q_star` so Q* stays aligned. |
| **Tokenizer** | `dict → StepTokens` (`input_field`; `objective_fields=` is `action` / `reward` / `terminated` / `truncated`) |

Compose `train_transform = compose(stages=(augmenter, tokenizer))`.
`DataLoader(transform=train_transform)` maps each step and packs into a `TokenBatch`.
Live inference in `15_inference.ipynb` uses the tokenizer without the augmenter so chosen actions match the env.


In [ ]:
def on_episode_terminated(ctx):
    return "episode_terminated" in ctx and bool(ctx["episode_terminated"])

def on_episode_truncated(ctx):
    return "episode_truncated" in ctx and bool(ctx["episode_truncated"])

def on_group_start(ctx):
    return bool(ctx["group_start"])

def on_reward_nonzero(ctx):
    return "reward" in ctx and ctx["reward"] != 0.0

def on_episode_start(ctx):
    return bool(ctx.get("episode_start")) | bool(ctx["group_start"])

# Pipeline order: augmenter → tokenizer

group_prefix = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {MAX_TASK_EPISODES} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At the end of each line, predict the value of all actions. Format:\n"
    "action,observation[,r=reward][,terminated][,truncated][,episode N].\n"
)

augmenter = Augmenter(
    seed_field="task_index",
    fields=[
        {
            "type": "discrete",
            "input_field": "action",
            "input_vector_field": "info_q_star",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "type": "discrete",
            "input_field": "observation",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
)

tokenizer = Tokenizer(
    input_fields=[
        {
            "type": "text",
            "format": group_prefix,
            "when": on_group_start,
        },
        {
            "type": "text",
            "input_field": "action",
            "format": "{field}",
        },
        {
            "type": "text",
            "input_field": "observation",
            "format": ",{field}",
        },
        {
            "type": "text",
            "input_field": "reward",
            "format": ",r={field:g}",
            "when": on_reward_nonzero,
        },
        {
            "type": "text",
            "input_field": "episode_terminated",
            "format": ",terminated",
            "when": on_episode_terminated,
        },
        {
            "type": "text",
            "input_field": "episode_truncated",
            "format": ",truncated",
            "when": on_episode_truncated,
        },
        {
            "type": "text",
            "input_field": "episode_index",
            "format": ",episode {field+1}",
            "when": on_episode_start,
        },
        {
            "type": "token",
            "format": "\n",
            "head_output": True,
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
        },
        {
            "input_field": "reward",
        },
        {
            "input_field": "terminated",
        },
        {
            "input_field": "truncated",
        },
    ],
    pretrained=PRETRAINED,
)

train_transform = compose(stages=(augmenter, tokenizer))

loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    batch_size=1,
    sample_field="task_index",
    transform=train_transform,
    prefetch=4,
    num_workers=0,
)


## Build The Model

A Mouse Core `Model` has a backbone and heads:

- `TransformerBackbone(pretrained=...)` loads the checkpoint including `embed_tokens`, looks up the packed `__text__` ids, and runs the decoder. Step templates and field packing live on `Tokenizer` only. Three arguments are required and describe how it runs on this machine rather than what it is, so they are not saved with the model and `load_model` asks for them again: `train_kernel` for the uncached forward (`"flex"`, block-sparse FlexAttention), `decode_kernel` for cached decode (`"flex"`, paged FlexAttention) and `dtype` for the base weights (`torch.float32` so the whole backbone trains in fp32). `model.to(device)` only moves. `use_norm` (required, saved with the model) keeps (`True`) or drops (`False`) the final RMSNorm.
- `RegressionHead` predicts one value per discrete action. `use_norm` (required) keeps (`True`) or drops (`False`) the head's input RMSNorm. `propagate_gradient` in `[0, 1]` scales backbone gradient through the pooled hidden state (`1` full flow, `0` none).
- `ConstantHead` is the centering constant. `forward` ignores the backbone state and returns `scale * value`. `value` starts at 0, so the output starts at 0. `scale` sets how far one step of `value` moves the output. The backup uses the delayed copy. The `reward_center` loss trains the online copy.

The backbone exposes `hidden_dim`, and the head uses that same value so the pieces connect cleanly.

`Model(...)` wraps the pieces behind a single forward call that returns predictions, objective data, and an optional cache. Pass `heads=` as a dict that names each head (`{"action_value": head, "reward_center": center}` here). `out.predictions["action_value"]` is the Q tensor; `out.predictions["reward_center"]` is the scalar `c`. `action_source="action_value"` is the name `get_action` reads.

### What packed steps look like

Three steps of episode 0 (each text field is its own tokenize run; reward gated off when zero; `episode_terminated` and `episode_truncated` emit only when true; `episode_index` emits on `episode_start` or group start):

```text
step 0:  action=0, observation=1, reward=0.0, episode_terminated=False, episode_index=0, episode_start=True
step 1:  action=2, observation=5, reward=0.0, episode_terminated=False, episode_index=0, episode_start=False
step 2:  action=1, observation=7, reward=1.0, episode_terminated=True, episode_index=0, episode_start=False
```

Each field fills its own `format=` (`{field}` for action, `,{field}` for observation, `,r={field:g}` for reward, `,terminated` for episode_terminated and `,truncated` for episode_truncated, `,episode {field+1}` for episode_index, and the const `\n` for `value`). `{field:g}` drops a trailing `.0`. Reward uses `on_reward_nonzero`. `episode_terminated` and `episode_truncated` emit only when the bool is true. `episode_index` uses `when=on_episode_start`, so `,episode` emits when `episode_start` is true or at group start. A later episode in the same task emits on its own first step.
The `when=on_group_start` const is the FrozenLake game / strategy / step-format blurb inserted once at the start of the `task_index` segment (here all three steps share task 0).
`*\n*` marks the head-output token (the row-ending newline) — the head reads Q from it to score the next action.

```text
task=0
Your job is to predict the future sum of rewards in FrozenLake. Navigate a grid; reach the goal for reward; a hole ends the episode with none. Each episode lasts at most 30 steps. You have 20 episodes to solve the task. The grid is permuted, so squares are not in order; action ids may be remapped.
Strategy: explore; keep a mental map of what has and has not been explored; avoid holes you have already fallen in; once you have a path to the goal, repeat it.
At the end of each line, predict the value of all actions. Format:
action,observation[,r=reward][,terminated][,truncated][,episode N].
0,1,episode 1*\n*
2,5*\n*
1,7,r=1,terminated*\n*
```

Packed in order: group prefix · step0 · step1 · step2. `head_output_indices` points at every `*\n*` token.


In [ ]:
backbone = TransformerBackbone(
    train_kernel="flex",
    decode_kernel="flex",
    dtype=torch.float32,
    use_norm=True,
    pretrained=PRETRAINED,
)

head = RegressionHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
    use_norm=True,
    propagate_gradient=1.0,
)

center = ConstantHead(scale=0.1)

model = Model(
    backbone=backbone,
    heads={"action_value": head, "reward_center": center},
    action_source="action_value",
    reasoner=None,
).train().to(device)
print(model)


## Training Phase

Each outer cycle runs `TRAIN_STEPS` optimizer updates via `run_train`. Mouse Core abstractions do most of the work:

1. `inputs, objective_data, group_id = loader.next_batch()` fills one batch (`batch_size=1`) with whole tasks (episode-0 start through a mouse-gym `terminated` or `truncated`) until the next task would pass `TOKEN_BUDGET` packed tokens.
2. `model(inputs)` embeds the `TokenBatch`, runs the backbone with per-sequence causal attention/RoPE, and produces flat per-step head predictions.
3. `objective(objective_data=objective_data, group_id=group_id, predictions=q, delayed_predictions=q_target, reward_center=out.predictions["reward_center"], delayed_reward_center=delayed_out.predictions["reward_center"])` computes the DQN loss and metrics. Pass `out.predictions["action_value"]` — the name given in `heads=`. The online scalar `c` is `out.predictions["reward_center"]`. The backup uses the delayed copy.
4. `AdamW` steps the heads at `LR_HEADS` and the backbone at `LR_BACKBONE`. The backbone and heads are fp32 (`dtype=torch.float32`), so every update lands in fp32 with no master weights.
5. Delayed Q and the delayed center come from the delayed model: `delayed_model = model.copy(heads=True, backbone=True, reasoner=False)` copies every head and the fp32 backbone (including token embeddings). After the online forward, `delayed_model(inputs)` runs the same `TokenBatch` through the delayed model under `torch.no_grad()`. `model_polyak(online=model, delayed=delayed_model, tau_heads=POLYAK_TAU_HEADS, tau_backbone=POLYAK_TAU_BACKBONE)` interpolates each copied section toward the online model after the optimizer step: `0` keeps it frozen, `1` copies the online weights (no delay). Every interpolated parameter is fp32, so a small `tau` is never rounded away. That includes the delayed center.

`DqnObjective` takes `discount`, called with the unpacked `objective_data` columns (`terminated` / `truncated` are bools). `boundary_discount` is the standard lookup: `gamma_step` always multiplies, then a terminated extra and a truncated extra (`1` when that flag is false). A true flag multiplies its extra; `0.0` zeros the whole term. `reward=None` and `value=None` skip those callables (raw `reward` column, raw Q). Pass `boundary_reward` / `boundary_value` to transform them. Required `temperature` is the SAC / soft-Q `α`: `0` is hard max-Q; `> 0` bootstraps from `α logsumexp(Q / α)` (same meaning as `get_action(temperature=)`). Required `double`: `False` reads that bootstrap from delayed Q; `True` is Double DQN, where detached online Q chooses the action and delayed Q scores it. Required `gate` is the continuation, like `reward` / `value` / `discount`. `gate=None` is the one-step target. `lambda_gate(td_lambda=)` is the λ-return to the end of the run. `nstep_gate(n=)` is the n-step return. `watkins_gate()` cuts where the taken action is not an online max. `value_gap_gate(beta=, normalize=, policy_delayed=, value_delayed=, bias=, eps=)` is the optimality-gap cut. `policy_delayed` picks `a* = argmax` from online or delayed Q. `value_delayed` scores `V = Q(s, a*)` and the taken action on online or delayed Q. Either flag, or both, reads the delayed net. The signed gap is `Q(s, a_taken) - V`. `normalize=True` divides by `max_a Q - min_a Q + eps` on the value Q (`eps` required). `normalize=False` uses the raw signed gap and takes no `eps`. `bias` shifts the gap, then it is capped at `0`. `c = exp(beta * gap)` (`14_train_offline_value_gap_dqn.ipynb`). `general_gate(gates=)` multiplies those matrices, so a step continues only where every gate continues. For any number of steps, the last step of a task is not updated when its target depends on the next step's value. It is updated when the target does not: a terminated or truncated γ of 0, or a horizon that puts no weight on that value. Required `cross_group_backups`: `"bootstrap"` fills in `V` on the step before the group boundary (the end of the batch, or a `group_id` break — a chunk boundary, time limit, or truncation whose rest was not sampled), so that step is updated from `V` and stays in the loss and in logged metrics. `"ignore"` leaves a backup that would use `V` past that boundary out of the target, and leaves that step out of the loss and out of logged metrics when the factor on that value is non-zero. `"fault"` raises on that step instead. A terminated or truncated γ of 0, or a horizon that puts no weight on that value, does not depend on it, so the step stays and `"fault"` does not raise. An earlier step whose backup stays inside the task is updated either way.

Pass `out.predictions["reward_center"]` as `reward_center=` and `delayed_out.predictions["reward_center"]` as `delayed_reward_center=` (`None` for both leaves the residual plain). The backup uses the delayed center for both potentials: one step is `G = r - c' + γ c' + γ V`. `action_value` trains Q. `reward_center` is `-c · mean(δ)` with `δ` detached, and `AdamW` steps `c` with the other heads. When `c = c'`, this telescopes to the same offset `c` on every action value, so greedy actions never change. With this notebook's `γ` (`1.0` in-task, `0` at task end) that offset is subtracted only on the task-final transition. `model_polyak` tracks `c'`.


In [ ]:
optimizer = AdamW(
    params=[
        {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
        {"params": model.heads.parameters(), "lr": LR_HEADS},
    ],
    lr=LR_BACKBONE,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-8,
)
delayed_model = model.copy(heads=True, backbone=True, reasoner=False)
objective = DqnObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
    temperature=0.0,
    double=True,
    cross_group_backups="fault",
    gate=None,
)

def run_train(
    *,
    model: Model,
    delayed_model: Model,
    optimizer: AdamW,
    objective: DqnObjective,
    loader: DataLoader,
    num_steps: int,
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Run ``num_steps`` optimizer steps on batches from ``loader``."""
    model.train()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data, group_id = loader.next_batch()
        out = model(inputs)
        with torch.no_grad():
            delayed_out = delayed_model(inputs)
        batch = to_device(data=objective_data, device=device)
        losses, metrics = objective(
            objective_data=batch,
            group_id=group_id.to(device),
            predictions=out.predictions["action_value"],
            delayed_predictions=delayed_out.predictions["action_value"],
            reward_center=out.predictions["reward_center"],
            delayed_reward_center=delayed_out.predictions["reward_center"],
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model_polyak(
            online=model,
            delayed=delayed_model,
            tau_heads=POLYAK_TAU_HEADS,
            tau_backbone=POLYAK_TAU_BACKBONE,
        )
    assert loss is not None
    return (loss, metrics)


## Run

Each of `NUM_CYCLES` cycles calls `run_train(num_steps=TRAIN_STEPS)`. Score the checkpoint later in `15_inference.ipynb`.


In [ ]:
for cycle in range(NUM_CYCLES):
    loss, metrics = run_train(
        model=model,
        delayed_model=delayed_model,
        optimizer=optimizer,
        objective=objective,
        loader=loader,
        num_steps=TRAIN_STEPS,
    )
    delta = metrics["in_run_delta"]
    assert isinstance(delta, torch.Tensor)
    print(f"cycle={cycle} train  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}  mean_delta={float(delta.mean()):.3f}  c={metrics['reward_center']:.3f}  c_delayed={metrics['delayed_reward_center']:.3f}")

loader.close()


## Push To The Hub

`push_model_to_hub` uploads the model to `MODEL_ID` and the tokenizer packing spec to a different repo (`TOKENIZER_ID`). Later, `load_model` reconstructs the `Model` and `load_tokenizer` on the tokenizer repo reloads the packing spec — formats, when-gates, and `head_output` cannot be recovered from the backbone alone.


In [ ]:
model.eval().to("cpu")
model_url, tokenizer_url = push_model_to_hub(
    model=model,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    tokenizer_repo_id=TOKENIZER_ID,
    private=False,
    clear=True,
)
print(f"Pushed model to {model_url}\nPushed tokenizer to {tokenizer_url}")
